# 02. Эксперименты — обучение рекомендателя

Пайплайн: `prepare` → `features` → `train`.

**Финальная модель v7:** LightGBM per-product + hybrid (`min_positives=100`) + early stopping по последнему месяцу train.

| Артефакт | Путь |
|----------|------|
| Модель | `models/model.bin` |
| Метрики | `models/metrics.json` |
| Журнал | `suggestions_04.md` |
| Реестр | `models/experiments_registry.json` |

MLflow (опционально): `python scripts/start_mlflow.py`, эксперимент `bank-product-recommender`.

Ячейки prepare/features/train тяжёлые — по умолчанию достаточно **загрузить уже посчитанные метрики**. Переобучение — опциональные ячейки ниже.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path("..").resolve()
CONFIG_PATH = ROOT / "configs" / "train.yaml"
with CONFIG_PATH.open(encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("ROOT:", ROOT)
print("random_seed:", config["random_seed"])
print("sample train/valid per month:",
      config["data"]["clients_per_month_train"],
      config["data"]["clients_per_month_valid"])
print("min_positives:", config["model"].get("min_positives_train"))
print("early_stopping:", config["model"].get("early_stopping"))
print("experiment:", config["mlflow"]["experiment_name"])
print("train_run:", config["mlflow"].get("train_run_name"))


def run_module(module: str, *extra: str) -> None:
    cmd = [sys.executable, "-m", module, "--config", str(CONFIG_PATH), *extra]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)

## 1. Результаты текущего `models/metrics.json` (v7)

Сравнение с popularity baseline на том же valid.

In [ ]:
metrics_path = ROOT / config["paths"]["metrics_path"]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
base = metrics["baseline"]
model = metrics["lightgbm"]
lift = model["map@7"] / base["map@7"] - 1

summary = pd.DataFrame([
    {"model": "popularity", **{k: base[k] for k in ("map@7", "precision@7", "recall@7")}},
    {"model": "lightgbm_v7", **{k: model[k] for k in ("map@7", "precision@7", "recall@7")}},
])
display(summary)
print(f"lift MAP@7 vs baseline: {lift:+.2%}")
print(f"n_train={metrics.get('n_train')}, n_valid={metrics.get('n_valid')}, n_features={metrics.get('n_features')}")
print(f"min_positives={metrics.get('min_positives_train')}, early_stopping={metrics.get('early_stopping')}")
print("skipped → popularity:", metrics.get("skipped_products"))
print("model.bin exists:", (ROOT / config["paths"]["model_path"]).exists())

## 2. AUC по продуктам (eligible-клиенты без продукта на t)

In [ ]:
aucs = pd.Series(metrics.get("product_aucs") or {}, dtype="float64").dropna().sort_values(ascending=False)
display(aucs.to_frame("roc_auc").head(15))
iters = {k: v for k, v in (metrics.get("best_iterations") or {}).items() if v is not None}
print("best_iterations (early stopping):", dict(sorted(iters.items(), key=lambda x: -x[1])[:8]), "...")

## 3. Краткая история экспериментов

Подробности и код — в `suggestions_04.md`.

In [ ]:
reg_path = ROOT / "models" / "experiments_registry.json"
if reg_path.exists():
    reg = json.loads(reg_path.read_text(encoding="utf-8"))
    rows = []
    for r in reg.get("runs", []):
        rows.append({
            "id": r.get("id"),
            "name": r.get("name"),
            "sample": r.get("sample"),
            "map@7": r.get("map@7"),
            "lift_vs_baseline_%": r.get("lift_vs_baseline_pct"),
            "best": r.get("kept_as_best"),
        })
    display(pd.DataFrame(rows))
    print("best_run:", reg.get("best_run"))
else:
    print("нет", reg_path)

## 4. Выводы для сдачи

1. **Baseline** popularity: MAP@7 ≈ 0.021.
2. **v7** (hybrid + ES + сэмпл 100k): MAP@7 ≈ **0.0257**, lift ≈ **+24%**.
3. SPW и полные лаги портфеля не улучшили MAP@7 — не вошли в финал.
4. Редкие продукты отдаём popularity (`min_positives_train=100`).
5. Сервис: `POST /recommend` читает `model.bin` + `data/serving/clients_features.parquet`.

## 5. (Опционально) Переобучить пайплайн

Раскомментируйте / выполните, если нужно пересобрать данные и модель. Долго на полном сэмпле.

In [ ]:
# run_module("src.data.prepare")
# run_module("src.features.build")
# run_module("src.models.train", "--skip-mlflow")  # или без --skip-mlflow при поднятом MLflow
print("Пропуск переобучения — используем готовые models/metrics.json")